# Assignment: Word Vectors, RNNs, and LSTMs

**CISC 6000 Deep Learning version**  
**Total:** 100 points

## Learning goals

After completing this assignment, you should be able to:

1. explain why learned word vectors are more useful than one-hot vectors;
2. create center–context pairs for the Word2Vec skip-gram task;
3. trace a recurrent neural network through a short sequence;
4. use the input, forget, output, and candidate values of an LSTM cell;
5. compare how vanilla RNNs and LSTMs represent sequence information.

## What you are **not** asked to do

You will **not** derive backpropagation through time, train Word2Vec from scratch, or implement LSTM backpropagation. The emphasis is on the main ideas and forward computations introduced in the lecture slides.

Complete every numbered `TODO`, run the public checks, and answer the short-response questions. NumPy is the only required package.

## Grading rubric

| Part | Topic | Points |
|---|---|---:|
| 0 | Setup and reproducibility | 5 |
| 1 | Word representations and Word2Vec | 30 |
| 2 | RNN forward computation and character LM | 35 |
| 3 | LSTM memory and gates | 30 |
| **Total** |  | **100** |

Most code tasks require only 1–6 lines. Public checks are intended to help you, not surprise you.

In [12]:
# Part 0 — Setup (5 points)
import numpy as np

SEED = 7
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)

# Change to True after completing the TODOs in a section.
RUN_TESTS = True

print("NumPy version:", np.__version__)
print("Random seed:", SEED)

NumPy version: 2.2.6
Random seed: 7


## Part 1 — Word representations and Word2Vec (30 points)

The slides contrast two representations:

- A **one-hot vector** has vocabulary size dimensions, with one 1 and all other entries 0. Different words are orthogonal, so the representation contains no natural notion of similarity.
- A learned **word vector/embedding** is dense. Words used in similar contexts can have nearby vectors.

Word2Vec learns embeddings through a proxy task. In **skip-gram**, the center word predicts nearby outside/context words. In **CBOW**, surrounding words predict the center word.

### 1.1 One-hot vectors (6 points)

Implement `one_hot`. Indices start at 0. The returned vector must contain floating-point values.

In [13]:
def one_hot(index, vocab_size):
    # TODO 1: create a zero vector, set one entry to 1, and return it.
    vec = np.zeros(vocab_size, dtype=float)
    vec[index] = 1.0
    return vec


if RUN_TESTS:
    assert np.array_equal(one_hot(2, 5), np.array([0., 0., 1., 0., 0.]))
    assert one_hot(0, 3).dtype.kind == "f"
    print("TODO 1 passed.")

TODO 1 passed.


### 1.2 Cosine similarity (8 points)

Cosine similarity measures the angle between two vectors:

\[
\cos(a,b)=\frac{a\cdot b}{\lVert a\rVert\lVert b\rVert}.
\]

The small embeddings below are supplied for exploration; they are not trained in this notebook.

In [14]:
words = ["king", "queen", "man", "woman", "apple", "orange"]
word_to_id = {w: i for i, w in enumerate(words)}

# Dimensions roughly represent [royalty, masculine, feminine, fruit].
embeddings = np.array([
    [1.00, 0.80, 0.05, 0.00],  # king
    [1.00, 0.05, 0.80, 0.00],  # queen
    [0.05, 1.00, 0.05, 0.00],  # man
    [0.05, 0.05, 1.00, 0.00],  # woman
    [0.00, 0.00, 0.00, 1.00],  # apple
    [0.00, 0.02, 0.00, 0.95],  # orange
], dtype=float)


def cosine_similarity(a, b):
    # TODO 2: compute and return cosine similarity.
    # You may assume neither vector is all zeros.
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


if RUN_TESTS:
    assert np.isclose(cosine_similarity(np.array([1., 0.]), np.array([2., 0.])), 1.0)
    assert np.isclose(cosine_similarity(np.array([1., 0.]), np.array([0., 1.])), 0.0)
    pairs = [("king", "queen"), ("king", "apple"), ("apple", "orange")]
    for a, b in pairs:
        score = cosine_similarity(embeddings[word_to_id[a]], embeddings[word_to_id[b]])
        print(f"cos({a:>6}, {b:<6}) = {score:.3f}")
    print("TODO 2 passed.")

cos(  king, queen ) = 0.658
cos(  king, apple ) = 0.000
cos( apple, orange) = 1.000
TODO 2 passed.


### 1.3 Skip-gram training pairs (10 points)

For a context window of 2, the center word `banking` in

`problems turning into banking crises as`

has the outside words `turning`, `into`, `crises`, and `as`. Implement the same idea for any token sequence. Return ordered `(center, context)` tuples.

In [15]:
def make_skipgram_pairs(tokens, window_size=2):
    pairs = []
    # TODO 3: for every center position, add nearby tokens within the window.
    # Do not pair a token position with itself.
    for i, center in enumerate(tokens):
        start = max(0, i - window_size)
        end = min(len(tokens), i + window_size + 1)
        for j in range(start, end):
            if j != i:
                pairs.append((center, tokens[j]))
    return pairs


if RUN_TESTS:
    toy = make_skipgram_pairs(["a", "b", "c"], window_size=1)
    assert toy == [("a", "b"), ("b", "a"), ("b", "c"), ("c", "b")]
    sentence = "problems turning into banking crises as".split()
    pairs = make_skipgram_pairs(sentence, window_size=2)
    banking_context = [outside for center, outside in pairs if center == "banking"]
    assert banking_context == ["turning", "into", "crises", "as"]
    print("Context of 'banking':", banking_context)
    print("TODO 3 passed.")

Context of 'banking': ['turning', 'into', 'crises', 'as']
TODO 3 passed.


### 1.4 Short response (6 points)

Answer in 3–5 sentences total.

1. Why are the one-hot vectors for `hotel` and `motel` unable to express their similarity?
2. In one sentence each, distinguish skip-gram from CBOW.
3. Why is predicting context words a useful proxy task for learning meaning?

> **Your response:** TODO

- The one-hot vectors for hotel and motel each have a single 1 in a different position, so their dot product, and therefore their cosine similarity, is always 0; nothing in the representation can encode that these two words mean almost the same thing. 

- Skip-gram uses the center word to predict each of its surrounding context words. CBOW does the reverse, using the surrounding context words together to predict the center word. 

- Predicting context is a useful proxy task because of the distributional hypothesis that states: words that appear in similar contexts tend to have similar meanings, so to predict context well the model has to give such words similar vectors.

## Part 2 — RNN forward computation (35 points)

An RNN processes a sequence one item at a time. The same parameters are reused at every time step:

\[
h_t=\tanh(x_tW_{xh}+h_{t-1}W_{hh}+b_h).
\]

The hidden state summarizes information received so far. Depending on the task, an RNN may produce one output for a whole sequence or an output at every step.

### 2.1 Character-level language-model data (7 points)

For the training text `hello`, the model receives `h,e,l,l` and should predict `e,l,l,o`. Convert both sequences to one-hot arrays.

In [16]:
vocab = ["h", "e", "l", "o"]
char_to_id = {ch: i for i, ch in enumerate(vocab)}


def encode_characters(text, char_to_id):
    # TODO 4: stack one_hot vectors for all characters in text.
    vocab_size = len(char_to_id)
    return np.stack([one_hot(char_to_id[ch], vocab_size) for ch in text])


if RUN_TESTS:
    X_chars = encode_characters("hell", char_to_id)
    Y_chars = encode_characters("ello", char_to_id)
    assert X_chars.shape == (4, 4)
    assert np.array_equal(np.argmax(X_chars, axis=1), [0, 1, 2, 2])
    assert np.array_equal(np.argmax(Y_chars, axis=1), [1, 2, 2, 3])
    print("Inputs:\n", X_chars)
    print("Target IDs:", np.argmax(Y_chars, axis=1))
    print("TODO 4 passed.")

Inputs:
 [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]
Target IDs: [1 2 2 3]
TODO 4 passed.


### 2.2 One RNN step (9 points)

Implement one hidden-state update. Notice that this is only a forward computation—no derivatives are required.

In [17]:
def rnn_step(x_t, h_prev, W_xh, W_hh, b_h):
    # TODO 5: implement the equation shown above with np.tanh.
    return np.tanh(x_t @ W_xh + h_prev @ W_hh + b_h)


if RUN_TESTS:
    x = np.array([1., 0.])
    h = np.array([0.2, -0.1])
    W_xh = np.eye(2)
    W_hh = 0.5 * np.eye(2)
    b_h = np.zeros(2)
    expected = np.tanh(np.array([1.1, -0.05]))
    assert np.allclose(rnn_step(x, h, W_xh, W_hh, b_h), expected)
    print("New hidden state:", rnn_step(x, h, W_xh, W_hh, b_h))
    print("TODO 5 passed.")

New hidden state: [ 0.8005 -0.05  ]
TODO 5 passed.


### 2.3 Unroll the RNN (9 points)

Apply `rnn_step` to every row of `X`. Start from an all-zero hidden state and return every hidden state in an array of shape `(T, H)`.

In [18]:
def rnn_forward(X, W_xh, W_hh, b_h):
    # TODO 6: initialize h, loop over time, and store each new h.
    h = np.zeros(W_hh.shape[0])
    states = []
    for x_t in X:
        h = rnn_step(x_t, h, W_xh, W_hh, b_h)
        states.append(h)
    return np.array(states)


if RUN_TESTS:
    X = np.eye(3)
    W_xh = np.array([[.4, .0], [.0, .4], [.2, .2]])
    W_hh = np.array([[.5, .0], [.0, .5]])
    b_h = np.zeros(2)
    states = rnn_forward(X, W_xh, W_hh, b_h)
    assert states.shape == (3, 2)
    assert np.allclose(states[0], np.tanh([.4, 0.]))
    print("Hidden states:\n", states)
    print("TODO 6 passed.")

Hidden states:
 [[0.3799 0.    ]
 [0.1877 0.3799]
 [0.2857 0.3713]]
TODO 6 passed.


### 2.4 Generate output probabilities (5 points)

The hidden state is mapped to vocabulary logits and then to probabilities with softmax. Complete the stable softmax function.

In [19]:
def softmax(logits):
    # TODO 7: subtract the maximum, exponentiate, and normalize.
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exps = np.exp(shifted)
    return exps / np.sum(exps, axis=-1, keepdims=True)


if RUN_TESTS:
    p = softmax(np.array([1., 2., 3.]))
    assert p.shape == (3,)
    assert np.isclose(p.sum(), 1.0)
    assert np.argmax(p) == 2
    assert np.isfinite(softmax(np.array([1000., 1001.]))).all()
    print("Probabilities:", p)
    print("TODO 7 passed.")

Probabilities: [0.09   0.2447 0.6652]
TODO 7 passed.


### 2.5 Short response (5 points)

Answer in 3–5 sentences total.

1. What information should (h_t) contain?
2. Why must the same RNN weights be used at every time step?
3. In a character language model, how is a generated character used to produce the next character?

> **Your response:** TODO
- The hidden state h_t should hold a compressed summary of everything in the sequence up to step t (x_1 through x_t) that matters for the task, so that it can be used to predict the next output. 

- The same weights (W_xh, W_hh, b_h) are reused at every time step so that the network applies one consistent rule wherever a pattern appears in the sequence no matter the sequence length. This keeps its parameter count independent of sequence length. 

- In a character language model, the output distribution at each step is used to pick the next character by being one-hot encoded and fed back in as the input at the next step, together with the updated hidden state, to produce the next prediction.

## Part 3 — LSTM memory and gates (30 points)

An LSTM separates the hidden state (h_t) from a cell state (c_t), which acts as memory. In simplified form:

\[
c_t=f_t\odot c_{t-1}+i_t\odot g_t,
\qquad
h_t=o_t\odot\tanh(c_t).
\]

- **Forget gate (f_t):** how much old memory to retain
- **Input gate (i_t):** how much new candidate information to write
- **Candidate (g_t):** proposed new content
- **Output gate (o_t):** how much cell content to expose as the hidden state

### 3.1 LSTM state update (15 points)

The gates are already assumed to have values in ([0,1]), while the candidate is in ([-1,1]). Implement only the two state equations above.

In [20]:
def lstm_state_update(c_prev, forget_gate, input_gate, candidate, output_gate):
    # TODO 8: compute c_t and h_t, then return both.
    c_t = forget_gate * c_prev + input_gate * candidate
    h_t = output_gate * np.tanh(c_t)
    return c_t, h_t


if RUN_TESTS:
    c_prev = np.array([2.0])

    # Retain old memory: f=1, i=0
    c_keep, h_keep = lstm_state_update(c_prev, np.array([1.]), np.array([0.]),
                                       np.array([.7]), np.array([1.]))
    assert np.allclose(c_keep, [2.0])

    # Erase old memory and write +0.5: f=0, i=1
    c_write, h_write = lstm_state_update(c_prev, np.array([0.]), np.array([1.]),
                                         np.array([.5]), np.array([1.]))
    assert np.allclose(c_write, [.5])
    assert np.allclose(h_write, np.tanh([.5]))
    print("Keep memory: c_t =", c_keep, "h_t =", h_keep)
    print("Replace memory: c_t =", c_write, "h_t =", h_write)
    print("TODO 8 passed.")

Keep memory: c_t = [2.] h_t = [0.964]
Replace memory: c_t = [0.5] h_t = [0.4621]
TODO 8 passed.


### 3.2 Interpret the gates (7 points)

For each scenario, choose sensible scalar gate values and calculate (c_t) and (h_t). You may use 0 or 1 for simple answers.

1. Preserve (c_{t-1}=0.8) without writing new information; expose the memory.
2. Erase (c_{t-1}=0.8) and write candidate (-0.5); expose the result.
3. Preserve (c_{t-1}=0.8) but hide it from the hidden state.

> **Your gate values and calculations:** TODO
1. Preserve c_{t−1} = 0.8, write nothing, expose memory: f = 1, i = 0, o = 1 (the value of g doesn't matter because i = 0)

- c_t = f·c_{t−1} + i·g = 1(0.8) + 0·g = 0.8
- h_t = o·tanh(c_t) = 1·tanh(0.8) ≈ 0.664

2. Erase c_{t−1} = 0.8, write candidate g = −0.5, expose result: f = 0, i = 1, o = 1

- c_t = 0(0.8) + 1(−0.5) = −0.5
- h_t = 1·tanh(−0.5) ≈ −0.462

3. Preserve c_{t−1} = 0.8 but hide it: f = 1, i = 0, o = 0

- c_t = 1(0.8) + 0·g = 0.8
- h_t = 0·tanh(0.8) = 0

Scenarios 1 and 3 store the same memory (c_t = 0.8). The only difference is the output gate, which controls whether that memory is shown in h_t. So an LSTM can keep information in its cell state without exposing it until it's needed.

### 3.3 Final comparison (8 points)

In 4–6 sentences, compare a vanilla RNN and an LSTM. Your response must mention parameter sharing, hidden state, cell state, gates, and why the LSTM can more easily preserve long-range information.

> **Your response:** TODO

 Both vanilla RNNs and LSTMs process a sequence one step at a time and use parameter sharing, reusing the same weights at every time step, so they can handle sequences of any length. A vanilla RNN stores everything it knows in a single hidden state, which is completely rewritten at each step by a matrix multiplication followed by tanh. 
 
 An LSTM adds a separate cell state that acts as long-term memory (conveyor belt), while its hidden state serves as the working output exposed at each step. The LSTM uses gates (forget, input, and output, each with values between 0 and 1) to decide how much old memory to keep (forget), how much new candidate information to write (input), and how much memory to reveal in the hidden state (output). 
 
 Setting the forget gate near 1 and the input gate near 0 lets information pass through many time steps almost unchanged, and gradients flow back along the same path without vanishing. In a vanilla RNN, by contrast, information and gradients pass through repeated multiplication by W_hh and tanh at every step, so they shrink quickly and long-range dependencies are hard to learn.

## Submission checklist

- [ ] Set `RUN_TESTS = True` and run the notebook from top to bottom.
- [ ] Complete all 8 numbered code TODOs.
- [ ] Complete all four short-response/calculation prompts.
- [ ] Keep the printed test outputs visible.
- [ ] Confirm that no cell raises an error after restarting the kernel.

### Optional challenge (+5 points)

Use the RNN functions to process the one-hot inputs for `hell`. Add a small hidden-to-output matrix, compute a probability distribution for every time step, and report the predicted next characters. No training or backpropagation is required.

In [21]:
# Optional challenge: run the RNN on "hell" and predict the next characters
X_hell = encode_characters("hell", char_to_id)

H = 4  # hidden size
V = len(vocab)

# Input -> hidden: each of h, e, l switches on its own hidden unit
W_xh = np.zeros((V, H))
W_xh[char_to_id["h"], 0] = 2.0
W_xh[char_to_id["e"], 1] = 2.0
W_xh[char_to_id["l"], 2] = 2.0

# Hidden -> hidden: unit 3 remembers "the previous character was e"
W_hh = np.zeros((H, H))
W_hh[1, 3] = 2.0
b_h = np.zeros(H)

# Hidden -> output (vocabulary logits)
W_hy = np.zeros((H, V))
W_hy[0, char_to_id["e"]] = 3.0   # after h, predict e
W_hy[1, char_to_id["l"]] = 3.0   # after e, predict l
W_hy[2, char_to_id["o"]] = 3.0   # after l, lean toward o ...
W_hy[3, char_to_id["l"]] = 6.0   # ... unless the char before was e -> predict l
b_y = np.zeros(V)

hidden_states = rnn_forward(X_hell, W_xh, W_hh, b_h)   # shape (4, H)
probs = softmax(hidden_states @ W_hy + b_y)            # shape (4, V), one row per step

predicted = [vocab[k] for k in np.argmax(probs, axis=1)]
for t, ch in enumerate("hell"):
    print(f"t={t}  input '{ch}'  probs {probs[t]}  ->  predicted '{predicted[t]}'")
print("Predicted next characters:", "".join(predicted))


t=0  input 'h'  probs [0.0475 0.8574 0.0475 0.0475]  ->  predicted 'e'
t=1  input 'e'  probs [0.0475 0.0475 0.8574 0.0475]  ->  predicted 'l'
t=2  input 'l'  probs [0.003  0.003  0.9401 0.0539]  ->  predicted 'l'
t=3  input 'l'  probs [0.0475 0.0475 0.0475 0.8574]  ->  predicted 'o'
Predicted next characters: ello
